# 05 — Ablation Study

On isole **chaque choix de design** pour mesurer son impact individuel.
C'est ce qu'on appelle une *ablation study* en ML : on retire ou fait varier
un composant à la fois en gardant tout le reste fixe.

**Axes étudiés :**
1. **Nombre de qubits** → dimension PCA, taille de l'espace quantique
2. **Bandwidth α** → paramètre de largeur du kernel
3. **Nombre de kernels** → combien en faut-il vraiment ?
4. **Type de kernel** → Fidelity vs Projected
5. **Concentration exponentielle** → évolution avec n_qubits

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

from src.preprocessing import QuantumScaler, FeatureReducer
from src.kernels import build_feature_map, build_quantum_kernel, compute_kernel_matrix
from src.kernels.kernel_matrix import kernel_statistics
from src.kernels.feature_maps import get_feature_map_library
from src.mkl.alignment import centered_alignment

import os
os.makedirs('../results', exist_ok=True)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42

In [ ]:
# Dataset de référence
N_SAMPLES = 200

X_raw, y = make_classification(
    n_samples=N_SAMPLES, n_features=20, n_informative=12,
    n_redundant=5, random_state=SEED, weights=[0.7, 0.3]
)
print(f'Dataset: {X_raw.shape}, Classes: {np.bincount(y)}')

## Ablation 1 : Impact du nombre de qubits

**Question** : est-ce que plus de qubits = meilleure performance ?

**Attendu** : performances stables ou légèrement croissantes jusqu'à un plateau,
puis chute due à la concentration exponentielle (si sans MKL).

In [ ]:
QUBIT_RANGE = [2, 4, 6, 8]  # ↑ = plus lent ; IBM teste jusqu'à 20
N_RUNS_ABL  = 5

def run_single_qubit_experiment(X_raw, y, n_qubits, method='mkl', n_runs=5):
    """Entraîne et évalue QMKL pour un n_qubits donné."""
    reducer = FeatureReducer(n_components=n_qubits)
    scaler  = QuantumScaler(feature_range=(0, 2))
    X = scaler.fit_transform(reducer.fit_transform(X_raw))

    fm_lib = get_feature_map_library(n_qubits)
    K_full = []
    for _, fm in fm_lib:
        qk = build_quantum_kernel(fm, kernel_type='fidelity')
        K_full.append(compute_kernel_matrix(qk, X))

    # Statistiques de concentration
    conc_stats = [kernel_statistics(K) for K in K_full]
    mean_of_means = np.mean([s['mean'] for s in conc_stats])
    mean_of_vars  = np.mean([s['variance'] for s in conc_stats])

    scores = []
    for seed in range(n_runs):
        idx = np.arange(len(y))
        idx_tr, idx_te = train_test_split(idx, test_size=0.33,
                                          random_state=seed, stratify=y)
        y_tr, y_te = y[idx_tr], y[idx_te]
        K_tr = [K[np.ix_(idx_tr, idx_tr)] for K in K_full]
        K_te = [K[np.ix_(idx_te, idx_tr)] for K in K_full]

        if method == 'mkl':
            w = centered_alignment(K_tr, np.outer(y_tr, y_tr).astype(float))
        else:  # single best
            Ky = np.outer(y_tr, y_tr).astype(float)
            aligns = [np.sum(K * Ky) / (np.linalg.norm(K,'fro') * np.linalg.norm(Ky,'fro') + 1e-10)
                      for K in K_tr]
            w = np.zeros(len(K_tr)); w[np.argmax(aligns)] = 1.0

        w = np.array(w); w = w / (w.sum() + 1e-10)
        Kc_tr = sum(wi * K for wi, K in zip(w, K_tr))
        Kc_te = sum(wi * K for wi, K in zip(w, K_te))

        ev = np.linalg.eigvalsh(Kc_tr)
        if ev.min() < 0:
            Kc_tr += (abs(ev.min()) + 1e-10) * np.eye(Kc_tr.shape[0])

        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(Kc_tr, y_tr)
        scores.append(roc_auc_score(y_te, svm.predict_proba(Kc_te)[:,1]))

    return {
        'mean': np.mean(scores), 'std': np.std(scores, ddof=1),
        'concentration_mean': mean_of_means,
        'concentration_var':  mean_of_vars,
    }

print('Testing n_qubits...')
abl_qubits_mkl    = {}
abl_qubits_single = {}

for nq in QUBIT_RANGE:
    print(f'  n_qubits={nq}')
    abl_qubits_mkl[nq]    = run_single_qubit_experiment(X_raw, y, nq, 'mkl',    N_RUNS_ABL)
    abl_qubits_single[nq] = run_single_qubit_experiment(X_raw, y, nq, 'single', N_RUNS_ABL)

print('Done!')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

dims  = QUBIT_RANGE
mkl_m = [abl_qubits_mkl[d]['mean']    for d in dims]
mkl_s = [abl_qubits_mkl[d]['std']     for d in dims]
sng_m = [abl_qubits_single[d]['mean'] for d in dims]
sng_s = [abl_qubits_single[d]['std']  for d in dims]

# ROC-AUC vs n_qubits
ax = axes[0]
ax.errorbar(dims, mkl_m, yerr=mkl_s, fmt='o-', label='QMKL-Centered', color='#2ecc71', capsize=5, linewidth=2)
ax.errorbar(dims, sng_m, yerr=sng_s, fmt='s--', label='Single-Best',   color='#e74c3c', capsize=5, linewidth=2)
ax.set_xlabel('Nombre de qubits')
ax.set_ylabel('ROC-AUC')
ax.set_title('Performance vs n_qubits')
ax.legend()

# Concentration : mean
ax = axes[1]
conc_means = [abl_qubits_mkl[d]['concentration_mean'] for d in dims]
ax.plot(dims, conc_means, 'o-', color='#3498db', linewidth=2)
ax.set_xlabel('Nombre de qubits')
ax.set_ylabel('Moyenne hors-diagonale du kernel')
ax.set_title('Concentration exponentielle (moyenne)')
ax.text(0.05, 0.05, '→ Tend vers 0\n  = concentration',
        transform=ax.transAxes, fontsize=9, color='red', alpha=0.7)

# Concentration : variance
ax = axes[2]
conc_vars = [abl_qubits_mkl[d]['concentration_var'] for d in dims]
ax.plot(dims, conc_vars, 's-', color='#9b59b6', linewidth=2)
ax.set_xlabel('Nombre de qubits')
ax.set_ylabel('Variance hors-diagonale du kernel')
ax.set_title('Concentration exponentielle (variance)')
ax.text(0.05, 0.05, '→ Tend vers 0\n  = perd discrimination',
        transform=ax.transAxes, fontsize=9, color='red', alpha=0.7)

plt.suptitle('Ablation — Impact du nombre de qubits', fontsize=13)
plt.tight_layout()
plt.savefig('../results/05_ablation_qubits.png', dpi=150, bbox_inches='tight')
plt.show()

## Ablation 2 : Impact du bandwidth α

**Question** : quelle valeur d'α donne les meilleures performances ?

**Attendu** : courbe en cloche — trop petit = pas de discrimination, trop grand = concentration.

In [ ]:
N_QUBITS_FIXED = 4
reducer = FeatureReducer(n_components=N_QUBITS_FIXED)
scaler  = QuantumScaler(feature_range=(0, 2))
X_fixed = scaler.fit_transform(reducer.fit_transform(X_raw))

ALPHA_RANGE = [0.2, 0.5, 1.0, 2.0, 4.0, 8.0, 14.0, 20.0]
FM_NAMES    = ['Z', 'ZZ']

abl_alpha = {fm: {} for fm in FM_NAMES}

for fm_name in FM_NAMES:
    print(f'\nFeature map: {fm_name}')
    for alpha in ALPHA_RANGE:
        fm = build_feature_map(fm_name, N_QUBITS_FIXED, alpha=alpha, reps=1)
        qk = build_quantum_kernel(fm, kernel_type='fidelity')
        K_full = compute_kernel_matrix(qk, X_fixed)
        stats  = kernel_statistics(K_full)

        scores = []
        for seed in range(5):
            idx = np.arange(len(y))
            idx_tr, idx_te = train_test_split(idx, test_size=0.33,
                                              random_state=seed, stratify=y)
            K_tr = K_full[np.ix_(idx_tr, idx_tr)]
            K_te = K_full[np.ix_(idx_te, idx_tr)]
            ev   = np.linalg.eigvalsh(K_tr)
            if ev.min() < 0:
                K_tr += (abs(ev.min()) + 1e-10) * np.eye(K_tr.shape[0])
            svm = SVC(kernel='precomputed', C=1.0, probability=True)
            svm.fit(K_tr, y[idx_tr])
            scores.append(roc_auc_score(y[idx_te], svm.predict_proba(K_te)[:,1]))

        abl_alpha[fm_name][alpha] = {
            'mean': np.mean(scores), 'std': np.std(scores, ddof=1),
            'k_mean': stats['mean'], 'k_var': stats['variance']
        }
        print(f'  α={alpha:5.1f}: AUC={np.mean(scores):.4f}, K_mean={stats["mean"]:.4f}, K_var={stats["variance"]:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Z': '#2ecc71', 'ZZ': '#e67e22'}

for ax_idx, (metric, ylabel, title) in enumerate([
    ('auc', 'ROC-AUC', 'Performance vs α'),
    ('kvar', 'Variance hors-diagonale', 'Concentration du kernel vs α'),
]):
    ax = axes[ax_idx]
    for fm_name in FM_NAMES:
        alphas = sorted(abl_alpha[fm_name].keys())
        if metric == 'auc':
            vals = [abl_alpha[fm_name][a]['mean'] for a in alphas]
            errs = [abl_alpha[fm_name][a]['std']  for a in alphas]
            ax.errorbar(alphas, vals, yerr=errs, fmt='o-',
                        label=fm_name, color=colors[fm_name], capsize=4, linewidth=2)
        else:
            vals = [abl_alpha[fm_name][a]['k_var'] for a in alphas]
            ax.plot(alphas, vals, 'o-', label=fm_name, color=colors[fm_name], linewidth=2)
    ax.set_xlabel('Bandwidth α', fontsize=12)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.set_xscale('log')

plt.suptitle('Ablation — Impact du bandwidth α', fontsize=13)
plt.tight_layout()
plt.savefig('../results/05_ablation_alpha.png', dpi=150, bbox_inches='tight')
plt.show()

## Ablation 3 : Combien de kernels faut-il ?

**Question** : est-ce que plus de kernels = toujours mieux ?

On ajoute les kernels un par un, du plus aligné au moins aligné.

In [ ]:
N_QUBITS_FIXED = 4
reducer = FeatureReducer(n_components=N_QUBITS_FIXED)
scaler  = QuantumScaler(feature_range=(0, 2))
X_fixed = scaler.fit_transform(reducer.fit_transform(X_raw))

fm_lib = get_feature_map_library(N_QUBITS_FIXED)
K_full = []
knames = []
for label, fm in fm_lib:
    qk = build_quantum_kernel(fm, kernel_type='fidelity')
    K_full.append(compute_kernel_matrix(qk, X_fixed))
    knames.append(label)

# Classer les kernels par alignement moyen (sur plusieurs splits)
Ky_full = np.outer(y, y).astype(float)
align_scores = []
for K in K_full:
    a = np.sum(K * Ky_full) / (np.linalg.norm(K,'fro') * np.linalg.norm(Ky_full,'fro') + 1e-10)
    align_scores.append(a)

ranked = np.argsort(align_scores)[::-1]
print('Kernels classés par alignement :')
for r in ranked:
    print(f'  {knames[r]:25s}: {align_scores[r]:.4f}')

# Test pour n_kernels = 1, 2, ..., N
n_kernel_results = {}

for n_k in range(1, len(K_full) + 1):
    subset = [K_full[i] for i in ranked[:n_k]]
    scores = []
    for seed in range(5):
        idx = np.arange(len(y))
        idx_tr, idx_te = train_test_split(idx, test_size=0.33,
                                          random_state=seed, stratify=y)
        y_tr, y_te = y[idx_tr], y[idx_te]
        K_sub_tr = [K[np.ix_(idx_tr, idx_tr)] for K in subset]
        K_sub_te = [K[np.ix_(idx_te, idx_tr)] for K in subset]

        if n_k == 1:
            w = [1.0]
        else:
            w = centered_alignment(K_sub_tr, np.outer(y_tr, y_tr).astype(float))
            w = np.array(w); w = w / (w.sum() + 1e-10)

        Kc_tr = sum(wi * K for wi, K in zip(w, K_sub_tr))
        Kc_te = sum(wi * K for wi, K in zip(w, K_sub_te))
        ev = np.linalg.eigvalsh(Kc_tr)
        if ev.min() < 0:
            Kc_tr += (abs(ev.min()) + 1e-10) * np.eye(Kc_tr.shape[0])

        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(Kc_tr, y_tr)
        scores.append(roc_auc_score(y_te, svm.predict_proba(Kc_te)[:,1]))

    n_kernel_results[n_k] = {'mean': np.mean(scores), 'std': np.std(scores, ddof=1),
                              'added': knames[ranked[n_k-1]]}
    print(f'n_kernels={n_k:2d} (+{knames[ranked[n_k-1]]:20s}): '
          f'{np.mean(scores):.4f} ± {np.std(scores,ddof=1):.4f}')

In [ ]:
n_ks  = sorted(n_kernel_results.keys())
means = [n_kernel_results[k]['mean'] for k in n_ks]
stds  = [n_kernel_results[k]['std']  for k in n_ks]
added = [n_kernel_results[k]['added'] for k in n_ks]

fig, ax = plt.subplots(figsize=(12, 5))
ax.errorbar(n_ks, means, yerr=stds, fmt='o-', color='#2980b9',
            capsize=5, linewidth=2, markersize=7)

# Annoter le kernel ajouté à chaque étape
for i, (n_k, m, name) in enumerate(zip(n_ks, means, added)):
    if i < len(n_ks) - 1:
        short = name.split('_')[0]  # juste le préfixe
        ax.annotate(f'+{short}', (n_k, m), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=7, color='#2c3e50')

# Marquer le maximum
best_n = n_ks[np.argmax(means)]
ax.axvline(best_n, color='red', linestyle='--', alpha=0.6,
           label=f'Optimal: {best_n} kernels')

ax.set_xlabel('Nombre de kernels combinés', fontsize=12)
ax.set_ylabel('ROC-AUC moyen', fontsize=12)
ax.set_title('Ablation — Impact du nombre de kernels (ajout séquentiel par alignement)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/05_ablation_n_kernels.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n→ Optimal : {best_n} kernels suffisent (AUC={max(means):.4f})')

## Ablation 4 : Fidelity vs Projected Kernel

**Question** : le kernel projeté est-il meilleur que le kernel de fidélité ?

**Attendu (IBM paper)** : PQK plus stable, moins concentré, meilleur en général.

In [ ]:
N_QUBITS_FIXED = 4
reducer = FeatureReducer(n_components=N_QUBITS_FIXED)
scaler  = QuantumScaler(feature_range=(0, 2))
X_fixed = scaler.fit_transform(reducer.fit_transform(X_raw))

kernel_types = ['fidelity', 'projected']
ktype_results = {}

for ktype in kernel_types:
    print(f'\nKernel type: {ktype}')
    fm_lib = get_feature_map_library(N_QUBITS_FIXED)
    K_full_list = []
    conc_stats  = []

    for label, fm in fm_lib:
        try:
            qk = build_quantum_kernel(fm, kernel_type=ktype, gamma=1.0)
            K  = compute_kernel_matrix(qk, X_fixed)
            K_full_list.append(K)
            conc_stats.append(kernel_statistics(K))
        except Exception as e:
            print(f'  Skipped {label}: {e}')

    scores = []
    for seed in range(5):
        idx = np.arange(len(y))
        idx_tr, idx_te = train_test_split(idx, test_size=0.33,
                                          random_state=seed, stratify=y)
        y_tr, y_te = y[idx_tr], y[idx_te]
        K_tr = [K[np.ix_(idx_tr, idx_tr)] for K in K_full_list]
        K_te = [K[np.ix_(idx_te, idx_tr)] for K in K_full_list]
        w = centered_alignment(K_tr, np.outer(y_tr, y_tr).astype(float))
        w = np.array(w); w = w / (w.sum() + 1e-10)
        Kc_tr = sum(wi * K for wi, K in zip(w, K_tr))
        Kc_te = sum(wi * K for wi, K in zip(w, K_te))
        ev = np.linalg.eigvalsh(Kc_tr)
        if ev.min() < 0:
            Kc_tr += (abs(ev.min()) + 1e-10) * np.eye(Kc_tr.shape[0])
        svm = SVC(kernel='precomputed', C=1.0, probability=True)
        svm.fit(Kc_tr, y_tr)
        scores.append(roc_auc_score(y_te, svm.predict_proba(Kc_te)[:,1]))

    ktype_results[ktype] = {
        'scores': scores,
        'mean': np.mean(scores),
        'std': np.std(scores, ddof=1),
        'conc_mean': np.mean([s['mean'] for s in conc_stats]),
        'conc_var':  np.mean([s['variance'] for s in conc_stats]),
    }
    print(f'  AUC: {np.mean(scores):.4f} ± {np.std(scores,ddof=1):.4f}')
    print(f'  Kernel mean: {ktype_results[ktype]["conc_mean"]:.4f}')
    print(f'  Kernel var:  {ktype_results[ktype]["conc_var"]:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {'fidelity': '#3498db', 'projected': '#e67e22'}

# Scores distribution
ax = axes[0]
for ktype, data in ktype_results.items():
    ax.boxplot(data['scores'], positions=[list(ktype_results.keys()).index(ktype)],
               widths=0.4, patch_artist=True,
               boxprops=dict(facecolor=colors[ktype], alpha=0.6))
ax.set_xticks(range(len(ktype_results)))
ax.set_xticklabels(list(ktype_results.keys()))
ax.set_ylabel('ROC-AUC')
ax.set_title('Distribution des scores')

# Concentration mean
ax = axes[1]
for i, (ktype, data) in enumerate(ktype_results.items()):
    ax.bar(i, data['conc_mean'], color=colors[ktype], alpha=0.8, edgecolor='black')
ax.set_xticks(range(len(ktype_results)))
ax.set_xticklabels(list(ktype_results.keys()))
ax.set_ylabel('Moyenne du kernel')
ax.set_title('Concentration moyenne')

# Concentration var
ax = axes[2]
for i, (ktype, data) in enumerate(ktype_results.items()):
    ax.bar(i, data['conc_var'], color=colors[ktype], alpha=0.8, edgecolor='black')
ax.set_xticks(range(len(ktype_results)))
ax.set_xticklabels(list(ktype_results.keys()))
ax.set_ylabel('Variance du kernel')
ax.set_title('Concentration variance\n(↑ = plus discriminant)')

plt.suptitle('Ablation — Fidelity vs Projected Quantum Kernel', fontsize=13)
plt.tight_layout()
plt.savefig('../results/05_ablation_kernel_type.png', dpi=150, bbox_inches='tight')
plt.show()